# Detection of Suspicious Transactions via Anomaly Detection (AML Project)

**Course:** Data Science  
**Author:** Slavena Georgieva
**Date:** April 2026

---

## 1. Introduction

Money laundering is the process of concealing the origins of illegally obtained money.
According to the United Nations, between **800 billion and 2 trillion** are laundered
globally each year — representing 2–5% of global GDP.

Traditional rule-based Anti-Money Laundering (AML) systems flag transactions based on
fixed thresholds (e.g. transactions above $10,000). While simple, these systems produce
high false-positive rates and are easily circumvented by criminals who deliberately
structure transactions to stay below detection thresholds.

This project takes a **statistical and machine learning approach** — using Anomaly Detection
to identify suspicious transactions based on their deviation from normal behaviour,
rather than fixed rules.

### Objectives
- Perform Exploratory Data Analysis (EDA) on two independent AML datasets
- Test a statistical hypothesis about transaction amounts
- Apply and compare three anomaly detection methods: Z-score, IQR, and Isolation Forest

## 2. Research Hypothesis

**Null Hypothesis (H₀):**  
There is no statistically significant difference between the transaction amounts
of normal and suspicious transactions.

**Alternative Hypothesis (H₁):**  
Suspicious transactions have significantly higher amounts compared to normal transactions.

$$H_0: \mu_{suspicious} = \mu_{normal}$$

$$H_1: \mu_{suspicious} > \mu_{normal}$$

This is a **one-tailed test** (right-tailed), since H₁ is directional — we expect
suspicious transactions to be *higher*, not just different.

We use a significance level of **α = 0.05**.

## 3. Data Sources & Description

### Sampling Strategy

Due to file size constraints (SAML-D: ~200MB, IBM HI-Large: ~15GB),
this project works with stratified samples of 10,000 rows from each dataset.

**Stratified sampling** ensures that the proportion of suspicious vs. normal
transactions is preserved from the original dataset — avoiding a sample
with zero suspicious transactions.

The samples are saved as separate CSV files and committed to the repository.
Full datasets are available on Kaggle (see README).

In [1]:
import pandas as pd
import numpy as np

# ── Load full SAML-D ───────────────────────────────────
saml_full = pd.read_csv('SAML-D.csv')

print(f"Full SAML-D shape: {saml_full.shape}")
print(f"Columns: {saml_full.columns.tolist()}")
print("\nLabel distribution:")
print(saml_full.iloc[:, -1].value_counts())


Full SAML-D shape: (9504852, 12)
Columns: ['Time', 'Date', 'Sender_account', 'Receiver_account', 'Amount', 'Payment_currency', 'Received_currency', 'Sender_bank_location', 'Receiver_bank_location', 'Payment_type', 'Is_laundering', 'Laundering_type']

Label distribution:
Laundering_type
Normal_Small_Fan_Out      3477717
Normal_Fan_Out            2302220
Normal_Fan_In             2104285
Normal_Group               528351
Normal_Cash_Withdrawal     305031
Normal_Cash_Deposits       223801
Normal_Periodical          210526
Normal_Plus_Mutual         155041
Normal_Mutual              125335
Normal_Foward               42031
Normal_single_large         20641
Structuring                  1870
Cash_Withdrawal              1334
Deposit-Send                  945
Smurfing                      932
Layered_Fan_In                656
Layered_Fan_Out               529
Stacked Bipartite             506
Behavioural_Change_1          394
Bipartite                     383
Cycle                         382

In [2]:
# ── Identify label column ──────────────────────────────
label_col = saml_full.columns[-1]  # usually last column

# ── Stratified sample (preserves suspicious ratio) ─────
saml_sample = saml_full.groupby(label_col, group_keys=False).apply(
    lambda x: x.sample(
        n=min(len(x), int(10_000 * len(x) / len(saml_full))),
        random_state=42
    )
).reset_index(drop=True)

print(f"Sample shape: {saml_sample.shape}")
print("\nLabel distribution in sample:")
print(saml_sample[label_col].value_counts())

# ── Save sample ─────────────────────────────────────────
saml_sample.to_csv('saml_sample.csv', index=False)
print("\n✓ Saved: saml_sample.csv")


Sample shape: (9985, 12)

Label distribution in sample:
Laundering_type
Normal_Small_Fan_Out      3658
Normal_Fan_Out            2422
Normal_Fan_In             2213
Normal_Group               555
Normal_Cash_Withdrawal     320
Normal_Cash_Deposits       235
Normal_Periodical          221
Normal_Plus_Mutual         163
Normal_Mutual              131
Normal_Foward               44
Normal_single_large         21
Cash_Withdrawal              1
Structuring                  1
Name: count, dtype: int64

✓ Saved: saml_sample.csv


C:\Users\Славена\AppData\Local\Temp\ipykernel_6260\2182074138.py:5: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  saml_sample = saml_full.groupby(label_col, group_keys=False).apply(


In [3]:
# ── Read IBM dataset in chunks (file is huge) ───────────
print("Reading IBM file in chunks... (this may take 1–2 minutes)")

chunks = []
for chunk in pd.read_csv(
    'HI-Large_Trans(1).csv',
    chunksize=100_000,
    dtype={'Account': str, 'Account.1': str},
    low_memory=False
):
    chunks.append(chunk)
    if len(chunks) == 5:  # read first ~500k rows
        break

ibm_full = pd.concat(chunks).reset_index(drop=True)

print(f"Loaded shape: {ibm_full.shape}")
print(f"Columns: {ibm_full.columns.tolist()}")


Reading IBM file in chunks... (this may take 1–2 minutes)
Loaded shape: (500000, 11)
Columns: ['Timestamp', 'From Bank', 'Account', 'To Bank', 'Account.1', 'Amount Received', 'Receiving Currency', 'Amount Paid', 'Payment Currency', 'Payment Format', 'Is Laundering']


In [4]:
# ── Parse HI-Large_Patterns.txt ───────────────────────
with open('HI-Large_Patterns.txt', 'r') as f:
    content = f.read()

rows = []
for line in content.split('\n'):
    line = line.strip()
    if line.startswith('BEGIN') or line.startswith('END') or line == '':
        continue
    parts = line.split(',')
    if len(parts) >= 11:
        rows.append(parts)

patterns_df = pd.DataFrame(rows, columns=[
    'Timestamp', 'From Bank', 'Account', 'To Bank', 'Account.1',
    'Amount Received', 'Receiving Currency', 'Amount Paid',
    'Payment Currency', 'Payment Format', 'Is Laundering'
])
patterns_df['Is Laundering'] = 1

print(f"Suspicious от Patterns: {len(patterns_df)}")

# ── 5000 suspicious + 5000 normal ─────────────────────
suspicious = patterns_df.sample(n=5000, random_state=42)
normal = ibm_full[ibm_full['Is Laundering'] == 0].sample(n=5000, random_state=42)

ibm_final = pd.concat([normal, suspicious]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"IBM final shape: {ibm_final.shape}")
print(f"Distribution:\n{ibm_final['Is Laundering'].value_counts()}")

ibm_final.to_csv('ibm_sample_final.csv', index=False)
print("✓ Saved: ibm_sample_final.csv")

Suspicious от Patterns: 137936
IBM final shape: (10000, 11)
Distribution:
Is Laundering
1    5000
0    5000
Name: count, dtype: int64
✓ Saved: ibm_sample_final.csv


In [5]:
# ── Зареди финалните sample файлове ───────────────────
saml = pd.read_csv('saml_sample_final.csv')
ibm = pd.read_csv('ibm_sample_final.csv')

print("=== SAML-D Sample ===")
print(f"Shape: {saml.shape}")
print(f"Columns: {saml.columns.tolist()}")
print(f"Suspicious: {saml['Is_laundering'].sum():.0f} ({saml['Is_laundering'].mean()*100:.1f}%)")

print("\n=== IBM HI-Large Sample ===")
print(f"Shape: {ibm.shape}")
print(f"Columns: {ibm.columns.tolist()}")
print(f"Suspicious: {ibm['Is Laundering'].sum()} ({ibm['Is Laundering'].mean()*100:.1f}%)")

=== SAML-D Sample ===
Shape: (14873, 12)
Columns: ['Time', 'Date', 'Sender_account', 'Receiver_account', 'Amount', 'Payment_currency', 'Received_currency', 'Sender_bank_location', 'Receiver_bank_location', 'Payment_type', 'Is_laundering', 'Laundering_type']
Suspicious: 9873 (66.4%)

=== IBM HI-Large Sample ===
Shape: (10000, 11)
Columns: ['Timestamp', 'From Bank', 'Account', 'To Bank', 'Account.1', 'Amount Received', 'Receiving Currency', 'Amount Paid', 'Payment Currency', 'Payment Format', 'Is Laundering']
Suspicious: 5000 (50.0%)


In [6]:
# Вземи всички suspicious + 5000 normal от пълния SAML-D
susp = saml_full[saml_full['Is_laundering'] == 1.0]
normal = saml_full[saml_full['Is_laundering'] == 0.0].sample(n=5000, random_state=42)

saml_final = pd.concat([susp, normal]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Shape: {saml_final.shape}")
print(f"Distribution:\n{saml_final['Is_laundering'].value_counts()}")

saml_final.to_csv('saml_sample_final.csv', index=False)
print("✓ Saved: saml_sample_final.csv")

Shape: (14873, 12)
Distribution:
Is_laundering
1    9873
0    5000
Name: count, dtype: int64
✓ Saved: saml_sample_final.csv


In [9]:
saml = pd.read_csv('saml_sample_final.csv')
ibm = pd.read_csv('ibm_sample_final.csv')

print("=== SAML-D Sample ===")
print(f"Shape: {saml.shape}")
print(f"Suspicious: {saml['Is_laundering'].sum():.0f} ({saml['Is_laundering'].mean()*100:.1f}%)")

print("\n=== IBM HI-Large Sample ===")
print(f"Shape: {ibm.shape}")
print(f"Suspicious: {ibm['Is Laundering'].sum()} ({ibm['Is Laundering'].mean()*100:.1f}%)")

=== SAML-D Sample ===
Shape: (14873, 12)
Suspicious: 9873 (66.4%)

=== IBM HI-Large Sample ===
Shape: (10000, 11)
Suspicious: 5000 (50.0%)
